In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
!ls /content/drive/MyDrive/microfinance_credit_scoring/data/train.csv

/content/drive/MyDrive/microfinance_credit_scoring/data/train.csv


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/microfinance_credit_scoring/data/train.csv")
df.head()


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [ ]:
df.shape


(614, 13)

In [ ]:
df.columns


Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object')

In [ ]:
# Target (what we want to predict)
y = df["Loan_Status"]

# Inputs (all other columns)
X = df.drop("Loan_Status", axis=1)


In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (614, 12)
y shape: (614,)


In [ ]:
df.isnull().sum()


,0
Loan_ID,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14


In [ ]:
# Fill numerical columns with median
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

# Fill categorical columns with mode
cat_cols = X.select_dtypes(include=["object"]).columns
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])


In [ ]:
X.isnull().sum()


,0
Loan_ID,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,0


In [ ]:
X.select_dtypes(include=["object"]).columns


Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Property_Area'],
      dtype='object')

In [ ]:
X = df[
    [
        "ApplicantIncome",
        "CoapplicantIncome",
        "LoanAmount",
        "Loan_Amount_Term",
        "Credit_History"
    ]
]



In [ ]:
import joblib

joblib.dump(X.columns.tolist(), "columns.pkl")


['columns.pkl']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
X_train.shape, X_test.shape


((491, 5), (123, 5))

In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
joblib.dump(pipeline, "credit_pipeline.pkl")

['credit_pipeline.pkl']

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Predict using pipeline
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.7886178861788617
              precision    recall  f1-score   support

           N       0.95      0.42      0.58        43
           Y       0.76      0.99      0.86        80

    accuracy                           0.79       123
   macro avg       0.85      0.70      0.72       123
weighted avg       0.83      0.79      0.76       123



In [ ]:
probabilities = pipeline.predict_proba(X_test)


In [ ]:
credit_scores = 300 + (probabilities[:, 1] * 600)


In [ ]:
credit_scores[:10]


array([776.9941043 , 788.29477612, 779.34912179, 794.53456778,
       794.6560965 , 782.01281519, 789.41432651, 795.02539744,
       777.26535982, 778.46669541])

In [ ]:
def get_risk_level(score):
    if score >= 780:
        return "Low Risk"
    elif score >= 700:
        return "Moderate Risk"
    else:
        return "High Risk"


In [ ]:
risk_levels = [get_risk_level(score) for score in credit_scores]
risk_levels[:10]


['Moderate Risk',
 'Low Risk',
 'Moderate Risk',
 'Low Risk',
 'Low Risk',
 'Low Risk',
 'Low Risk',
 'Low Risk',
 'Moderate Risk',
 'Moderate Risk']

In [ ]:
def get_eligibility(risk):
    if risk == "Low Risk":
        return "Eligible"
    elif risk == "Moderate Risk":
        return "Manual Review"
    else:
        return "Not Eligible"


In [ ]:
eligibility = [get_eligibility(risk) for risk in risk_levels]
eligibility[:10]


['Manual Review',
 'Eligible',
 'Manual Review',
 'Eligible',
 'Eligible',
 'Eligible',
 'Eligible',
 'Eligible',
 'Manual Review',
 'Manual Review']

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Credit_Score": credit_scores,
    "Risk_Level": risk_levels,
    "Eligibility": eligibility
})

results.head(10)


,Credit_Score,Risk_Level,Eligibility
0,776.994104,Moderate Risk,Manual Review
1,788.294776,Low Risk,Eligible
2,779.349122,Moderate Risk,Manual Review
3,794.534568,Low Risk,Eligible
4,794.656097,Low Risk,Eligible
5,782.012815,Low Risk,Eligible
6,789.414327,Low Risk,Eligible
7,795.025397,Low Risk,Eligible
8,777.265360,Moderate Risk,Manual Review
9,778.466695,Moderate Risk,Manual Review
